# Avaliação T038 — adaptador LoRA × modelo sem treino

Processa as imagens de **teste** (`data/processed/treino/test.jsonl`, 81 páginas do `br-doc-extraction`) com os dois modelos do Ollama, mesma instrução, temperatura 0 e esquema em `format`, e compara com o rótulo convertido. Desenho: [`2026-11-26-treino-lora-t038-design.md`](../docs/superpowers/specs/2026-11-26-treino-lora-t038-design.md).

Pré-requisitos: `uv run python -m visiondoc.treino.preparar` já executado e o Ollama ativo. Modelo ausente é pulado (dá para rodar só o *zero-shot* antes do treino). As métricas são provisórias até a T037; `bbox` e `face` não são avaliados (rótulos fixos).

O relatório vai para `data/processed/avaliacao-t038.json` (ignorado pelo git). As saídas deste notebook só mostram identificadores e métricas; mesmo assim, limpe-as antes de commitar.

In [ ]:
import io, json, time
from pathlib import Path

import ollama
import pandas as pd
from PIL import Image

from visiondoc.treino.avaliacao import CLASSES, acerto_por_campo, acuracia_tipo, f1_macro, matriz_confusao
from visiondoc.treino.esquema import SaidaVLM
from visiondoc.treino.instrucao import INSTRUCAO, TEXTO_USUARIO, hash_instrucao

RAIZ = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
PASTA_IMAGENS = RAIZ / "data" / "raw" / "imagens"
TESTE = RAIZ / "data" / "processed" / "treino" / "test.jsonl"
RELATORIO = RAIZ / "data" / "processed" / "avaliacao-t038.json"

HOST = "http://localhost:11434"  # somente máquina local (ADR-009)
MODELOS = {"sem_treino": "qwen3.5:4b", "adaptador": "visiondoc-qwen35-4b-lora"}
AMOSTRA = 3  # None = as 81 páginas (~25 min por modelo nesta GPU)
LADO_MAXIMO = 1280

client = ollama.Client(host=HOST)
instalados = {m.model: m.digest for m in client.list().models}
modelos = {alias: nome for alias, nome in MODELOS.items() if nome in instalados}
print("avaliando:", modelos, "| ausentes:", [n for n in MODELOS.values() if n not in instalados])

## Teste e rótulos

In [ ]:
exemplos = [json.loads(l) for l in TESTE.read_text("utf-8").splitlines()]
if AMOSTRA:
    exemplos = exemplos[:AMOSTRA]
for ex in exemplos:
    real = json.loads(ex["messages"][2]["content"])["documentos"][0]
    ex["tipo_real"], ex["campos_real"] = real["tipo"], real["campos"]
print(len(exemplos), "páginas de teste")

## Inferência (mesmos parâmetros para os dois modelos)

In [ ]:
def jpeg(caminho: Path) -> bytes:
    img = Image.open(caminho).convert("RGB")
    img.thumbnail((LADO_MAXIMO, LADO_MAXIMO))
    buf = io.BytesIO()
    img.save(buf, "JPEG", quality=90)
    return buf.getvalue()


def inferir(modelo: str, imagem: bytes) -> tuple[str, float]:
    t = time.perf_counter()
    r = client.chat(
        model=modelo,
        messages=[
            {"role": "system", "content": INSTRUCAO},
            {"role": "user", "content": TEXTO_USUARIO, "images": [imagem]},
        ],
        format=SaidaVLM.model_json_schema(),
        think=False,
        options={"temperature": 0, "num_predict": 1024},
    )
    return r.message.content, round(time.perf_counter() - t, 2)


resultados = {alias: [] for alias in modelos}
for alias, nome in modelos.items():  # um modelo por vez: evita trocar de modelo na GPU de 4 GB
    for i, ex in enumerate(exemplos, 1):
        texto, segundos = inferir(nome, jpeg(PASTA_IMAGENS / ex["imagem"]))
        try:
            doc = SaidaVLM.model_validate_json(texto).documentos[0]
            tipo, campos, valido = doc.tipo, doc.campos.model_dump(), True
        except Exception:
            tipo, campos, valido = "invalido", {}, False
        resultados[alias].append(
            {"id": ex["id"], "tipo_real": ex["tipo_real"], "tipo_previsto": tipo,
             "json_valido": valido, "segundos": segundos,
             "campos_previstos": campos, "campos_reais": ex["campos_real"]}
        )
        print(f"[{alias} {i}/{len(exemplos)}] {ex['id']}: {segundos} s, válido={valido}")

## Métricas por modelo

In [ ]:
def resumir(linhas: list[dict]) -> dict:
    reais = [l["tipo_real"] for l in linhas]
    previstos = [l["tipo_previsto"] for l in linhas]
    por_campo: dict[str, list[int]] = {}
    extras = 0
    for l in linhas:
        r = acerto_por_campo(l["campos_previstos"], l["campos_reais"])
        extras += len(r["extras"])
        for campo, ok in r["rotulados"].items():
            acertos = por_campo.setdefault(campo, [0, 0])
            acertos[0] += ok
            acertos[1] += 1
    return {
        "n": len(linhas),
        "acuracia_tipo": round(acuracia_tipo(previstos, reais), 4),
        "f1_macro": round(f1_macro(previstos, reais), 4),
        "classes": list(CLASSES),
        "matriz_confusao": matriz_confusao(previstos, reais),
        "json_valido": round(sum(l["json_valido"] for l in linhas) / len(linhas), 4),
        "segundos_medio": round(sum(l["segundos"] for l in linhas) / len(linhas), 2),
        "acerto_por_campo": {c: {"acertos": a, "total": t, "taxa": round(a / t, 4)} for c, (a, t) in sorted(por_campo.items())},
        "campos_extras_nao_rotulados": extras,
    }


resumo = {alias: resumir(linhas) for alias, linhas in resultados.items()}
tabela = pd.DataFrame(
    {a: {k_: r[k_] for k_ in ("n", "acuracia_tipo", "f1_macro", "json_valido", "segundos_medio", "campos_extras_nao_rotulados")} for a, r in resumo.items()}
)
display(tabela)
campos = pd.DataFrame({a: {c: v["taxa"] for c, v in r["acerto_por_campo"].items()} for a, r in resumo.items()})
display(campos)
for a, r in resumo.items():
    print(a, "matriz (linhas = real, colunas = previsto,", r["classes"], "):", r["matriz_confusao"])

## Relatório

In [ ]:
relatorio = {
    "tarefa": "T038",
    "amostra": len(exemplos),
    "hash_instrucao": hash_instrucao(),
    "modelos": {a: {"nome": n, "digest": instalados[n]} for a, n in modelos.items()},
    "resumo": resumo,
    "paginas": resultados,
}
RELATORIO.write_text(json.dumps(relatorio, ensure_ascii=False, indent=2), "utf-8")
print("Relatório:", RELATORIO)